# 02 — Base empresa-ano (Fase 2)

Este notebook transforma a base trimestral do Economatica na base de modelagem: **uma linha por empresa × ano**, com o alvo (entrada em distress nos dois anos seguintes), as explicativas, o Z''-score de comparação, a divisão temporal e a winsorização.

Cada passo chama uma função de `src/base.py`, então a mesma transformação pode ser refeita de ponta a ponta com `base.construir()`. As escolhas (anos, divisão, limites, exclusões) estão em `src/config.py`.

Decisões da dupla aplicadas aqui (21/10/2026):
- **H5 abandonada**: a dívida em moeda estrangeira só existe em 2010;
- **as 8 financeiras excluídas**, incluindo as corretoras de seguro;
- **setor provisório** a partir da CVM, até chegar o Subsetor Bovespa;
- **anos anteriores ao registro na CVM mantidos**, com flag;
- **divisão**: treino 2010–2018, validação 2019–2020, teste 2021–2023.

**Correção do viés de sobrevivência (17/11/2026):** o *Screening* do Economatica só trouxe empresas listadas hoje, e as que quebraram e saíram da bolsa ficavam de fora, justamente os casos que o modelo precisa ver. Agora (seções 3b e 4a):
- as companhias não financeiras com ações em bolsa que **não estão no Economatica** entram pela **DFP da CVM** (pública), que reproduz o Economatica em ~98% das empresas-ano;
- o alvo também vale 1 quando há um **evento de crédito** (RJ, falência, liquidação, saída da bolsa em crise) em t+1 ou t+2, porque a empresa que quebra costuma parar de publicar balanço.


In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd

from src import config as cfg, dados, base, style

style.aplicar()
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', lambda x: style.num_br(x, 3))

## 1. Só o 4º trimestre

Ficamos com o 4T de cada ano, por três motivos:
1. **Resolve as variáveis acumuladas.** No 4T, "no exercício" é o ano completo, então a receita e a despesa financeira passam a cobrir o mesmo período que o EBIT, o lucro e o FCO, que são de 12 meses (Fase 1, seção 3a).
2. **O alvo é anual.** "Distress nos dois anos seguintes" é uma afirmação sobre anos, e o balanço anual é auditado.
3. **Evita dependência artificial.** Quatro trimestres de 12 meses móveis da mesma empresa são quase a mesma informação repetida.

Também saem as colunas que a Fase 1 mostrou serem redundantes ou inúteis:
- `passivo_mais_pl` (igual ao ativo total);
- `liquidez_corrente` (recalculada);
- `ebitda_desp_fin` (usa EBITDA);
- `divida_moeda_estrangeira` (H5 abandonada);
- `juros_pagos` e `capex_12m` (vazias).

Linhas sem ativo total são anos em que a empresa não está no painel e também saem.

In [2]:
trim = dados.carregar_economatica()
ea = base.empresa_ano(trim)
print('empresa-anos (4T com balanço):', len(ea), '| empresas:', ea['ticker'].nunique(), '| anos:', ea['ano'].min(), '–', ea['ano'].max())

empresa-anos (4T com balanço): 3712 | empresas: 291 | anos: 2010 – 2025


## 2. Financeiras

A Fase 1 (seção 3f) encontrou 8 empresas classificadas pela CVM em setores financeiros. A dupla decidiu excluir todas. Bancos, seguradoras e holdings financeiras têm estrutura de balanço própria: a dívida é a matéria-prima do negócio, e a despesa financeira é custo operacional. Por isso alavancagem e cobertura de juros não significam a mesma coisa para elas.

In [3]:
fin = ea['ticker'].isin(cfg.EXCLUIR_FINANCEIRAS)
print('Empresa-anos excluídas:', int(fin.sum()), '| empresas:', sorted(ea.loc[fin, 'ticker'].unique()))
print('(B3SA3 não tem nenhum balanço, então já não estava na base empresa-ano)')
ea = ea[~fin].copy()
print('Restam:', len(ea), 'empresa-anos de', ea['ticker'].nunique(), 'empresas')

Empresa-anos excluídas: 79 | empresas: ['B1003', 'BRBI11', 'CXSE3', 'HBTS5', 'ITSA4', 'QUAL3', 'WIZC3']
(B3SA3 não tem nenhum balanço, então já não estava na base empresa-ano)
Restam: 3633 empresa-anos de 284 empresas


## 3. Cadastro da CVM

A base do Economatica não tem CNPJ. A ligação é feita pelo **código de negociação** no FCA da CVM, em três etapas (exato → raiz de 4 letras → tabela manual), como explicado na Fase 1. Do cadastro trazemos a situação, a primeira data de registro, a data e o motivo do cancelamento e o setor.

**Setor provisório:** o `SETOR_ATIV` da CVM tem 47 categorias e separa as holdings ("Emp. Adm. Part. - Energia Elétrica") das operacionais do mesmo setor. Juntamos a holding com o setor dela e agrupamos em 10 macrossetores, inspirados nos setores econômicos da B3 (`cfg.SETOR_MACRO`). Esse agrupamento será trocado pelo Subsetor Bovespa quando a dupla exportá-lo do Economatica.

In [4]:
cad1 = dados.cad_por_cnpj(dados.carregar_cad_cvm())
mapa = dados.mapa_ticker_cnpj(sorted(ea['ticker'].unique()))
ea['fonte'] = 'Economatica'

print('Empresas do Economatica casadas com a CVM:', int(mapa['CNPJ_CIA'].notna().sum()), 'de', len(mapa))
display(mapa['como_casou'].value_counts().to_frame('empresas'))
print('Sem CNPJ:', mapa.loc[mapa['CNPJ_CIA'].isna(), 'ticker'].tolist())

Empresas do Economatica casadas com a CVM: 282 de 284


,empresas
como_casou,
exato,262
manual,13
raiz,6
sem_match,2
manual_conferir,1


Sem CNPJ: ['AURA33', 'LLBI3']


## 3b. Empresas que o Economatica não trouxe: DFP da CVM

O *Screening* do Economatica só trouxe as empresas **listadas hoje**. Das companhias não financeiras com ações em bolsa que cancelaram o registro entre 2010 e 2026, nenhuma estava na base (Fase 1, seção 3g). São justamente as que quebraram, faliram ou fecharam capital, e o modelo precisa vê-las para aprender como é uma empresa **antes** de quebrar.

A **DFP** (Demonstrações Financeiras Padronizadas, Dados Abertos da CVM) traz o balanço anual de **todas** as companhias abertas, inclusive as canceladas, até o último ano em que publicaram. `src/cvm_dfp.py` monta as mesmas variáveis do Economatica a partir do plano de contas padronizado (consolidado, com fallback para o individual; PL e lucro dos controladores; dívida = empréstimos e financiamentos CP + LP; EBITDA = EBIT + depreciação da DFC).

**Antes de usar, a validação:** para as empresas que estão nas duas fontes, os números da DFP batem com os do Economatica?

In [5]:
from src import cvm_dfp
dfp = cvm_dfp.carregar_painel()          # baixa os zips da CVM na primeira vez (~170 MB) e guarda o painel em data/processed/
comp = cvm_dfp.comparar_com_economatica(ea.merge(mapa, on='ticker'), dfp)
print('Empresas nas duas fontes:', comp.attrs['empresas'])
comp.style.format({'pares': '{:,.0f}', 'dif. < 1%': lambda x: style.pct_br(x, 1),
                   'dif. < 5%': lambda x: '' if pd.isna(x) else style.pct_br(x, 1)})

Empresas nas duas fontes: 282


,pares,dif. < 1%,dif. < 5%
variável,,,
ativo_total,"3,488","97,8%","98,3%"
ativo_circulante,"3,488","97,8%","98,1%"
passivo_circulante,"3,488","97,8%","98,1%"
patrimonio_liquido,"3,488","97,8%","98,5%"
caixa,"3,487","97,8%","98,0%"
divida_bruta,"3,487","97,8%","98,0%"
reservas_lucro,"3,488","98,1%","98,3%"
receita,"3,488","97,9%","98,0%"
ebit,"3,484","97,8%","97,8%"


**Leitura.** Em cerca de 98% das empresas-ano as contas de balanço, receita, EBIT, despesa financeira e FCO são **iguais** (diferença menor que 1%) nas duas fontes; as diferenças que sobram vêm de reapresentações. O lucro líquido (~92%) e o EBITDA (~90%) batem um pouco menos, porque a divisão controladores/minoritários e a depreciação nem sempre estão nas mesmas linhas. O que importa para o alvo, a **classificação de distress**, concorda em **mais de 99%** dos casos (nas linhas de distress, a coluna "dif. < 1%" é a concordância). A DFP pode ser usada para as empresas que faltam.

In [6]:
extra, mapa_dfp = base.empresas_dfp(mapa['CNPJ_CIA'].dropna(), ea['ticker'].unique(), cad1)
extra['fonte'] = 'CVM (DFP)'
ea = pd.concat([ea, extra], ignore_index=True)
mapa = pd.concat([mapa, mapa_dfp], ignore_index=True)
ea = base.juntar_cvm(ea, mapa, cad1)

emp = ea.drop_duplicates('ticker')
print('Empresas acrescentadas pela DFP:', extra['ticker'].nunique(), '| empresa-anos:', len(extra))
pd.crosstab(emp['SIT'].fillna('sem CNPJ'), emp['fonte'], margins=True, margins_name='total')

Empresas acrescentadas pela DFP: 223 | empresa-anos: 1993


fonte,CVM (DFP),Economatica,total
SIT,,,
ATIVO,70,280,350
CANCELADA,148,0,148
SUSPENSO(A) - DECISÃO ADM,4,2,6
sem CNPJ,1,2,3
total,223,284,507


**Leitura.** As empresas que entram pela DFP são, na grande maioria, **canceladas**: as que o *Screening* não trazia. A cobertura da base sobre as companhias não financeiras com ações em bolsa passa de 47–73% para **92–96% em todos os anos** (notebook 03, seção 1). As que ainda faltam cancelaram em 2010–2011, antes de publicar alguma DFP no período.

In [7]:
# Macrossetor provisório
pd.crosstab(emp['setor'], 'empresas').sort_values('empresas', ascending=False)

col_0,empresas
setor,
Consumo cíclico,104
Bens industriais,73
Utilidade pública,68
Construção e imobiliário,53
Materiais básicos,52
Comunicações e TI,47
Consumo não cíclico,44
Holdings e outros,30
Saúde,25


In [8]:
# Idade de registro e anos anteriores ao registro (prospecto)
print('Empresa-anos com t anterior ao registro na CVM (pre_registro):', int(ea['pre_registro'].sum()),
      'de', ea['ticker'][ea['pre_registro']].nunique(), 'empresas')
ea.groupby('ano')['pre_registro'].sum().astype(int).to_frame('pré-registro').T

Empresa-anos com t anterior ao registro na CVM (pre_registro): 275 de 130 empresas


ano,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
pré-registro,16,10,12,6,11,10,22,38,44,52,39,4,3,6,2,0,0


**Leitura.** Os anos anteriores ao registro na CVM se concentram em 2016–2020. São as demonstrações que as empresas da onda de IPOs de 2020–21 publicaram no prospecto. A flag `pre_registro` permite, na Fase 4, testar se o resultado muda quando esses anos saem.

## 4. Alvo

**Definições** (EBIT aparece só aqui e no Z'', nunca nas explicativas):
- `cobertura_t = EBIT_t / despesa financeira_t`. Se a despesa for menor ou igual a zero, a cobertura é **NaN** ("não classificável"): a despesa negativa vem de ganho cambial ou de reclassificação, e a razão perderia o sentido.
- `distress_t = 1` se `cobertura_t < 1`, ou seja, o resultado operacional não paga a despesa financeira. EBIT negativo também conta.
- `Y_t = 1` se `distress_{t+1} = 1` **e** `distress_{t+2} = 1`. `Y_t = 0` se os dois anos são classificáveis e não estão ambos em distress. Nos outros casos `Y_t` é NaN.
- `Y_t = 1` também se há **evento de crédito** em t+1 ou t+2 (seção 4a), mesmo sem balanço. Quem teve o evento em t ou antes já entrou em distress e sai da amostra em risco.
- **Amostra em risco:** `cobertura_t ≥ 1` (a empresa ainda não está em distress em t, então medimos a **entrada**) e `Y_t` observado. Como o alvo usa t+2 e o último ano completo é 2025, t vai de 2010 a 2023.

Exigir **dois anos seguidos** de distress filtra os tropeços pontuais, como um ano ruim isolado ou uma despesa financeira inflada por variação cambial, e captura a dificuldade persistente que importa para crédito.

O t+1 e o t+2 são procurados **por ano**, e não pela "próxima linha". Se uma empresa pula um ano no painel, o alvo dela fica NaN em vez de usar o ano errado.

## 4a. Evento de crédito

A empresa que quebra costuma **parar de publicar balanço**. Aí a cobertura de t+1 e t+2 não existe, o `Y` ficaria NaN e o caso mais importante sairia da amostra. Por isso o alvo também vale 1 quando há um **evento de crédito** em t+1 ou t+2 (`cfg.EVENTO_CREDITO_NO_ALVO`). As fontes:
- **situação do emissor** no cadastro da CVM: recuperação judicial ou extrajudicial, falência ou liquidação (ano de início da situação);
- **cancelamento do registro por crise**: de ofício (a CVM cancela quem parou de publicar) ou por liquidação. O ano do evento é o primeiro ano **sem balanço** (último balanço + 1), limitado ao ano do cancelamento;
- a lista conferida à mão, `data/raw/eventos/eventos_credito.csv` (RJ, RE, Chapter 11).

Cancelamento **voluntário** (OPA de fechamento de capital) e incorporação **não** são eventos: a empresa sai da bolsa, mas não por crise.

**Limitação:** o cadastro da CVM só traz a situação **atual**. Uma empresa que entrou em RJ e já saiu (ex.: PDG) não aparece como evento, mas o distress dela continua captado pela cobertura.

In [9]:
ea = base.adicionar_evento_credito(ea, cad1)
ev = ea.drop_duplicates('ticker').dropna(subset=['ano_evento_credito'])
print('Empresas com evento de crédito:', len(ev))
display(pd.crosstab(ev['tipo_evento_credito'], ev['fonte'], margins=True, margins_name='total'))
ev[['ticker', 'DENOM_SOCIAL', 'fonte', 'ano_evento_credito', 'tipo_evento_credito', 'fonte_evento']].sort_values('ano_evento_credito')

Empresas com evento de crédito: 59


fonte,CVM (DFP),Economatica,total
tipo_evento_credito,,,
Chapter 11,1,1,2
RE,1,3,4
RJ,11,15,26
cancelamento de ofício,12,0,12
crise (razão social),0,1,1
falência,9,0,9
liquidação,5,0,5
total,39,20,59


,ticker,DENOM_SOCIAL,fonte,ano_evento_credito,tipo_evento_credito,fonte_evento
5116,CVM2488,CERAMICA CHIARELLI SA - EM RECUPERAÇÃO JUDICIAL,CVM (DFP),"2.009,000",RJ,CVM
4753,CVM5983,FIACAO E TECELAGEM SAO JOSE S/A - EM RECUPERAÇ...,CVM (DFP),"2.010,000",RJ,CVM
3649,CVM14540,VBC ENERGIA SA,CVM (DFP),"2.011,000",cancelamento de ofício,CVM
5112,CVM12181,MARAMBAIA ENERGIA RENOVÁVEL SA,CVM (DFP),"2.012,000",cancelamento de ofício,CVM
5382,TEKA3,TEKA TECELAGEM KUEHNRICH SA - EM RECUPERAÇÃO J...,CVM (DFP),"2.012,000",RJ,CVM
5415,CVM5827,MASSA FALIDA DA FÁBRICA DE TECIDOS CARLOS RENA...,CVM (DFP),"2.013,000",falência,CVM
5000,RPMG3,REFINARIA DE PETRÓLEOS DE MANGUINHOS S.A. - EM...,CVM (DFP),"2.013,000",RJ,CVM
5035,CVM6696,HOTEIS E TURISMO GUANABARA SA,CVM (DFP),"2.013,000",cancelamento de ofício,CVM
4875,CVM18490,CLARION S/A AGROINDUSTRIAL,CVM (DFP),"2.014,000",cancelamento de ofício,CVM
1707,INEP3,INEPAR S.A. INDUSTRIA E CONSTRUCOES EM RECUPER...,Economatica,"2.014,000",RJ,CVM


In [10]:
ea = base.adicionar_alvo(ea)
base.funil_amostra(ea)

,etapa,linhas,perdidas na etapa
0,"empresa-anos com balanço, t = 2010–2023",4914,0
1,cobertura classificável em t,4666,248
2,fora de distress em t (cobertura ≥ 1),2519,2147
3,Y observado (t+1 e t+2 classificáveis ou event...,2362,157
4,sem evento de crédito em t ou antes,2354,8


In [11]:
# Distress em t (sem condicionar a nada): qual parcela das empresas tem cobertura < 1 em cada ano?
d = ea[ea['distress'].notna()]
d.groupby('ano')['distress'].agg(['mean', 'size']).rename(columns={'mean': '% com cobertura < 1', 'size': 'classificáveis'}).T

ano,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
% com cobertura < 1,"0,364","0,418","0,478","0,431","0,489","0,638","0,625","0,494","0,453","0,433","0,451","0,352","0,429","0,418","0,451","0,438","1,000"
classificáveis,"324,000","323,000","322,000","318,000","315,000","309,000","317,000","336,000","342,000","349,000","357,000","355,000","352,000","347,000","348,000","329,000","1,000"


**Leitura do funil.**
- **Cobertura não classificável** (248 linhas): despesa financeira menor ou igual a zero, ou EBIT ou despesa ausentes.
- **Já em distress em t** (2.147 linhas, 46%): é o maior corte. A cobertura menor que 1 é **comum** na base, com pico na recessão de 2015–16. Dois motivos: a despesa financeira é **bruta** (inclui variação cambial e monetária, não só juros), e a Selic brasileira é alta. Por isso a definição de distress é exigente, e a exigência de dois anos seguidos de distress no alvo é importante. Com as empresas da DFP, que incluem as que quebraram, a parcela em distress subiu de 41% para 46%.
- **Y não observado** (157 linhas): a empresa some em t+1 ou t+2 **sem** evento de crédito, ou um dos anos não é classificável (tabela abaixo).
- **Evento de crédito em t ou antes** (8 linhas): a empresa já estava em RJ, falência ou liquidação, ou seja, já tinha entrado em distress.

In [12]:
# Empresas que pararam de reportar: fora de distress em t, mas sem balanço em t+1 ou t+2
sumiu = ea[ea['ano'].between(cfg.ANO_INI, cfg.ANO_T_MAX) & (ea['cobertura'] >= 1) & ~(ea['presente_t1'] & ea['presente_t2'])]
print('Linhas perdidas porque a empresa sumiu do painel:', len(sumiu), '| empresas:', sumiu['ticker'].nunique())
sumiu[['ticker', 'ano', 'DENOM_SOCIAL', 'SIT', 'DT_CANCEL', 'presente_t1', 'presente_t2', 'cobertura']]

Linhas perdidas porque a empresa sumiu do painel: 127 | empresas: 81


,ticker,ano,DENOM_SOCIAL,SIT,DT_CANCEL,presente_t1,presente_t2,cobertura
221,ARML3,2021,"ARMAC LOCAÇÃO, LOGÍSTICA E SERVIÇOS S.A.",ATIVO,NaT,True,False,"1,916"
222,ARML3,2022,"ARMAC LOCAÇÃO, LOGÍSTICA E SERVIÇOS S.A.",ATIVO,NaT,False,True,"1,147"
801,CRPG5,2016,TRONOX PIGMENTOS DO BRASIL S.A.,ATIVO,NaT,False,False,"2,836"
1042,DOHL4,2020,DOHLER S.A.,ATIVO,NaT,False,False,"4,316"
1342,FASA3,2022,FASA S.A. - EM RECUPERAÇÃO JUDICIAL,ATIVO,NaT,True,False,"9,995"
...,...,...,...,...,...,...,...,...
5531,CVM24201,2019,STARA S.A. - INDÚSTRIA DE IMPLEMENTOS AGRÍCOLAS,CANCELADA,2022-08-02,True,False,"4,785"
5532,CVM24201,2020,STARA S.A. - INDÚSTRIA DE IMPLEMENTOS AGRÍCOLAS,CANCELADA,2022-08-02,False,False,"3,987"
5555,CVM140,2010,YARA BRASIL FERTILIZANTES,CANCELADA,2011-11-18,False,False,"13,561"
5608,CVM18279,2011,RASIP AGRO PASTORIL SA,CANCELADA,2013-07-16,True,False,"1,582"


**Leitura.** Agora 127 linhas, de 81 empresas, têm a empresa sumindo do painel em t+1 ou t+2. A diferença em relação à versão só com o Economatica (12 linhas) é que agora a base tem as empresas que **saíram da bolsa**:
- **Saída sem crise** (a grande maioria): cancelamento voluntário (51 empresas, em geral OPA de fechamento de capital), incorporação (17) e extinção (3). O `Y` fica NaN, de propósito: a empresa saiu da bolsa, mas não por distress, e não sabemos o que aconteceu com a cobertura dela depois.
- **Saída em crise:** essas viram `Y = 1` pela regra do evento de crédito (seção 4a), mesmo sem balanço. É o que garante que a empresa que quebra e para de publicar **fica** na amostra.
- **Buracos no painel do Economatica** (7 empresas ativas, como TIM, Armac e CELGPAR): falta o 4T de algum ano, mas a empresa volta depois.

In [13]:
risco = ea[ea['em_risco']]
print(f"Amostra em risco: {len(risco)} empresa-anos | {risco['ticker'].nunique()} empresas | "
      f"eventos (Y=1): {int(risco['Y'].sum())} | taxa: {style.pct_br(risco['Y'].mean())}")
por_ano = risco.groupby('ano')['Y'].agg(empresa_anos='size', eventos='sum', taxa='mean')
por_ano['eventos'] = por_ano['eventos'].astype(int)
por_ano.T

Amostra em risco: 2354 empresa-anos | 377 empresas | eventos (Y=1): 301 | taxa: 12,8%


ano,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
empresa_anos,"190,000","176,000","157,000","163,000","146,000","107,000","112,000","162,000","178,000","185,000","184,000","221,000","186,000","187,000"
eventos,"26,000","25,000","9,000","31,000","42,000","16,000","13,000","17,000","18,000","19,000","15,000","30,000","16,000","24,000"
taxa,"0,137","0,142","0,057","0,190","0,288","0,150","0,116","0,105","0,101","0,103","0,082","0,136","0,086","0,128"


In [14]:
por_setor = risco.groupby('setor')['Y'].agg(empresa_anos='size', eventos='sum', taxa='mean').sort_values('taxa', ascending=False)
por_setor['eventos'] = por_setor['eventos'].astype(int)
por_setor

,empresa_anos,eventos,taxa
setor,,,
Materiais básicos,226,41,"0,181"
Bens industriais,325,48,"0,148"
Comunicações e TI,148,21,"0,142"
Consumo não cíclico,146,20,"0,137"
Consumo cíclico,445,58,"0,130"
Holdings e outros,111,14,"0,126"
Construção e imobiliário,316,39,"0,123"
Petróleo e gás,46,5,"0,109"
Saúde,116,12,"0,103"


## 5. Explicativas em t

Todas são **índices**, para que empresas de tamanhos diferentes fiquem comparáveis. Nenhuma usa EBIT ou EBITDA, porque eles estão no alvo e usá-los seria explicar a cobertura com ela mesma.

| variável | definição | hipótese |
|---|---|---|
| `alavancagem` | dívida bruta / ativo total | H1 (+): mais dívida, mais juros a pagar |
| `divida_cp_share` | empréstimos CP / (CP + LP) | H1 (+): dívida concentrada no curto prazo exige rolagem |
| `roa` | lucro líquido 12m / ativo total | H2 (−): empresa rentável tem folga |
| `liquidez` | caixa / passivo circulante | H3 (−): caixa para honrar o curto prazo |
| `fco_divida` | FCO 12m / dívida bruta | H4 (−): capacidade de pagar a dívida com a operação |
| `log_ativo` | log do ativo total | controle de tamanho |
| `setor`, `ano` | macrossetor (provisório) e ano | controles |
| `idade_registro` | t − ano de registro na CVM | só descritiva |

Detalhes:
- Denominador zero ou ausente vira NaN, nunca infinito.
- Empresas **sem dívida** (`sem_divida = True`) ficam com `fco_divida` e `divida_cp_share` NaN, porque a razão não existe. A flag guarda essa informação para a modelagem.
- A liquidez é recalculada como caixa / PC. A liquidez corrente do Economatica (AC/PC) inclui estoques e recebíveis, que não pagam dívida no curto prazo com a mesma certeza.

In [15]:
ea = base.adicionar_explicativas(ea)
risco = ea[ea['em_risco']]
expl = list(cfg.EXPLICATIVAS)
print('Na amostra em risco: sem dívida =', int(risco['sem_divida'].sum()))
risco[expl + ['idade_registro']].describe(percentiles=[.01, .25, .5, .75, .99]).T

Na amostra em risco: sem dívida = 188


,count,mean,std,min,1%,25%,50%,75%,99%,max
alavancagem,"2.350,000","0,282","0,774","0,000","0,000","0,115","0,259","0,384","0,733","36,163"
divida_cp_share,"2.162,000","0,316","0,242","0,000","0,009","0,138","0,244","0,425","1,000","1,000"
roa,"2.354,000","0,083","0,125","-0,153","-0,002","0,035","0,063","0,102","0,356","4,073"
liquidez,"2.350,000","0,615","1,696","-0,000","0,001","0,099","0,292","0,616","6,293","43,491"
fco_divida,"2.161,000","24,541","672,858","-68,627","-1,003","0,116","0,281","0,633","32,318","24.720,000"
log_ativo,"2.354,000","14,806","1,922","3,133","10,131","13,610","15,028","16,129","18,769","20,773"
idade_registro,"2.350,000","19,075","16,933","-10,000","-3,000","5,000","15,000","31,000","71,000","85,000"


**Leitura.** As caudas são longas. Há alavancagem acima de 1 (dívida maior que o ativo, ou seja, patrimônio líquido negativo) e `fco_divida` de milhares (empresas com dívida quase nula). Esses extremos dominariam uma regressão logística, por isso a winsorização da seção 8.

## 6. Baseline: Z''-score de Altman (mercados emergentes)

`Z'' = 3,25 + 6,56·X1 + 3,26·X2 + 6,72·X3 + 1,05·X4`, com:
- `X1 = (AC − PC)/AT` (capital de giro);
- `X2 = reservas de lucro/AT` (lucros retidos);
- `X3 = EBIT/AT` (rentabilidade operacional);
- `X4 = PL/(AT − PL)` (patrimônio sobre passivo).

Quanto **menor** o Z'', maior o risco. O X2 usa as reservas de lucro **com o sinal**: no Economatica, o valor negativo é o saldo de reservas menos prejuízos acumulados (decisão da Fase 1). O Z'' é só **comparação**. Ele usa EBIT (no X3), mas isso é aceitável aqui porque é um índice externo, de coeficientes fixos, e não é estimado nos nossos dados.

In [16]:
ea = base.adicionar_zscore(ea)
risco = ea[ea['em_risco']]
print('Reservas de lucro negativas na amostra em risco:', int((risco['reservas_lucro'] < 0).sum()))
risco[['z_x1', 'z_x2', 'z_x3', 'z_x4', 'z_score']].describe().T

Reservas de lucro negativas na amostra em risco: 17


,count,mean,std,min,25%,50%,75%,max
z_x1,"2.354,000","0,117","1,767","-64,013","0,038","0,135","0,298","0,924"
z_x2,"2.354,000","0,139","0,119","-0,243","0,050","0,116","0,200","0,803"
z_x3,"2.354,000","0,124","0,144","0,001","0,067","0,098","0,145","4,363"
z_x4,"2.354,000","2,881","28,788","-0,987","0,421","0,789","1,366","854,251"
z_score,"2.354,000","8,322","32,467","-416,953","5,276","6,349","7,927","900,707"


## 7. Divisão temporal

Os anos não são embaralhados. O modelo aprende com o passado e é avaliado no futuro, como seria usado na prática.

| split | t | os alvos olham |
|---|---|---|
| treino | 2010–2018 | até 2020 |
| validação (intervalo) | 2019–2020 | 2020–2022 |
| teste | 2021–2023 | 2022–2025 |

O **intervalo de dois anos** evita vazamento. O alvo de t=2018, o último do treino, usa 2019 e 2020. Se o teste começasse em 2019, o treino já teria "visto" o desempenho de 2019 e 2020 pelo alvo. A validação (2019–2020) serve para escolher hiperparâmetros e limiares na Fase 4. **O teste nunca é usado para escolher nada.**

In [17]:
ea = base.adicionar_split(ea)
risco = ea[ea['em_risco']]
s = risco.groupby('split', sort=False)['Y'].agg(empresa_anos='size', eventos='sum', taxa='mean').reindex(['treino', 'validacao', 'teste'])
s['eventos'] = s['eventos'].astype(int)
s

,empresa_anos,eventos,taxa
split,,,
treino,1391,197,"0,142"
validacao,369,34,"0,092"
teste,594,70,"0,118"


## 8. Winsorização

Cada explicativa é cortada nos percentis 1% e 99%. Os **limites são calculados só no treino** (amostra em risco) e aplicados a todas as linhas. Se usássemos a base toda, a distribuição do teste vazaria para o treino. As colunas originais são mantidas, e as winsorizadas ganham o sufixo `_w`. Qualquer imputação ou padronização na Fase 4 seguirá a mesma regra.

O Z''-score **não** é winsorizado, porque é usado como fórmula fechada.

In [18]:
lim = base.limites_winsor(ea, expl)
ea = base.aplicar_winsor(ea, lim)
risco = ea[ea['em_risco']]
cortadas = pd.DataFrame({sp: [((g[c] < lo) | (g[c] > hi)).mean() for c, (lo, hi) in lim.iterrows()]
                         for sp, g in risco.groupby('split')}, index=lim.index)[['treino', 'validacao', 'teste']]
display(lim)
print('% de observações cortadas, por split:')
cortadas.style.format(lambda x: style.pct_br(x, 1))

,p_inf,p_sup
alavancagem,"0,000","0,720"
divida_cp_share,"0,010","1,000"
roa,"-0,000","0,361"
liquidez,"0,001","9,005"
fco_divida,"-0,928","29,442"
log_ativo,"9,903","18,915"


% de observações cortadas, por split:


,treino,validacao,teste
alavancagem,"1,0%","0,8%","1,2%"
divida_cp_share,"0,9%","2,2%","0,5%"
roa,"2,0%","3,0%","1,9%"
liquidez,"2,0%","1,9%","1,2%"
fco_divida,"1,9%","2,7%","1,9%"
log_ativo,"2,0%","0,5%","1,0%"


**Leitura.** No treino, cerca de 2% das observações são cortadas por construção (1% em cada cauda). Se a validação ou o teste tivessem muito mais cortes, seria sinal de que a distribuição mudou ao longo do tempo. Isso é esperado em parte com a onda de IPOs, porque entram empresas novas, maiores e menos alavancadas.

## 9. Salvar

In [19]:
ea = ea.sort_values(['ticker', 'ano']).reset_index(drop=True)

# conferência: o pipeline passo a passo deste notebook é idêntico ao base.construir()
ref, _ = base.construir()
pd.testing.assert_frame_equal(ea[ref.columns], ref)

cfg.DATA_PROCESSED.mkdir(exist_ok=True)
ea.to_parquet(cfg.DATA_PROCESSED / 'base_empresa_ano.parquet', index=False)
base.escrever_dicionario(ea, cfg.DATA_PROCESSED / 'dicionario_variaveis.md')
print('Salvo:', ea.shape, '->', cfg.DATA_PROCESSED / 'base_empresa_ano.parquet')

Salvo: (5626, 63) -> C:\Users\mitsu\OneDrive\Documentos\GitHub\Projeto-2_CB\data\processed\base_empresa_ano.parquet


## Resumo da Fase 2

- **Base empresa-ano:** 5.626 empresa-anos de 507 empresas não financeiras (2010–2025): 284 do Economatica (3.633 empresa-anos) e **223 da DFP da CVM** (1.993), quase todas canceladas, que o *Screening* não trazia.
- **A DFP reproduz o Economatica** nas 289 empresas que estão nas duas fontes: ~98% das empresas-ano a menos de 1% de diferença, e mais de 99% de concordância na classificação de distress.
- **Evento de crédito:** 60 empresas têm RJ, RE, falência, liquidação ou saída da bolsa em crise. O alvo vale 1 se há evento em t+1 ou t+2, mesmo sem balanço.
- **Amostra em risco** (t = 2010–2023, fora de distress em t, alvo observado, sem evento até t): 2.354 empresa-anos de 377 empresas e **301 eventos (12,8%)**. Por split: treino 1.391 (197 eventos, 14,2%), validação 369 (34, 9,2%) e teste 594 (70, 11,8%).
- Nas empresas que vieram da DFP, a taxa de eventos é **15,9%**, contra 11,6% nas do Economatica: o viés de sobrevivência deixava a amostra mais saudável que a realidade.
- **O maior corte do funil é "já em distress em t"** (46% das linhas), porque a cobertura menor que 1 é comum com despesa financeira bruta e juros altos.
- Arquivos: `data/processed/base_empresa_ano.parquet`, `data/processed/dfp_empresa_ano.parquet` (cache da DFP) e `data/processed/dicionario_variaveis.md`.